<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 컴퓨터 비전 · 09. DETR

## End-to-End Object Detection with Transformers

- **원 논문:** [End-to-End Object Detection with Transformers](https://arxiv.org/abs/2005.12872)
- **저자 / 발표:** Nicolas Carion et al. · ECCV 2020 (2020)
- **난이도 / 예상 시간:** 중급 · 약 120분
- **선수 지식:** Python, PyTorch tensor, 합성곱, 역전파의 기초
- **로컬 학습 데이터:** `data/field_curriculum/vision_shapes.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** object queries, bipartite matching, no-object class, set loss를 재현한다.

**축소하거나 생략한 부분:** ResNet-50, 100 queries, COCO 대신 작은 CNN, 3 queries, 단일 물체를 사용한다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §3.1 Eq. (1): Hungarian matching | Task 1 tiny_match | 최소 permutation |
| §3.2 Fig. 2: decoder object queries and image cross-attention | Task 2 explicit query self/cross-attention | [B,H,3,16] cross-attention weights |
| §3.1 Eq. (2) and §3.2: Hungarian set loss/no-object class | Task 3 | set loss; GT-independent objectness query로 held-out class/box 평가 |

## 핵심 아이디어와 수식

$$\hat{\sigma}=\arg\min_{\sigma}\sum_i\mathcal{L}_{match}(y_i,\hat{y}_{\sigma(i)})$$

**기호 해설:** σ는 prediction과 target의 일대일 배정, L_match는 class와 box matching cost다.

**코드 대응:** Task 1이 matching cost, Task 2가 query self-attention과 image cross-attention, Task 3이 set loss를 구현한다.

**입력과 출력 shape:** 입력 [B,1,16,16] → memory [B,16,24] → 3 query class [B,3,5], box [B,3,4].

**포트폴리오 구현 범위:** 단일 물체라 완전 Hungarian 알고리즘은 최소 query 선택으로 축소된다. 학습 matching은 GT를 쓰지만 평가는 no-object 확률만으로 query를 먼저 고른다.

아래 코드는 결과만 내는 라이브러리 호출로 핵심을 감추지 않는다. config, 논문 고유
연산, 모델의 `forward`, 목적함수, `train_step`, 평가 함수를 순서대로 직접 작성한다.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import itertools
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(20260814)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device


def locate(relative_path):
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / relative_path
        if candidate.exists():
            return candidate
    raise FileNotFoundError(relative_path)


dataset_path = locate("data/field_curriculum/vision_shapes.npz")
raw = np.load(dataset_path)
images = torch.from_numpy(raw["images"]).float()
labels = torch.from_numpy(raw["labels"]).long()
masks = torch.from_numpy(raw["masks"]).float()
boxes = torch.from_numpy(raw["boxes"]).float()
train_idx = torch.from_numpy(raw["train_idx"][:128]).long()
test_idx = torch.from_numpy(raw["test_idx"][:64]).long()
images, labels, masks, boxes = ACCELERATOR.move(
    images,
    labels,
    masks,
    boxes,
)
train_idx, test_idx = ACCELERATOR.move(train_idx, test_idx)
class_names = raw["class_names"].tolist()
x_train = images[train_idx]
y_train = labels[train_idx]
assert images.shape == (480, 1, 16, 16)
assert masks.shape == (480, 16, 16)
assert boxes.shape == (480, 4)
print(ACCELERATOR.summary())
print(dataset_path.name, images.shape, class_names)

### 구현 단계 1 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §3.1 Eq. (1): Hungarian matching
- **노트북에서 구현할 부분:** Task 1 tiny_match
- **구현 이유:** 논문의 계산 경로를 작은 tensor로 보존하면서 각 중간값을
  assertion으로 관찰하기 위해서다.
- **완료 증거:** 최소 permutation

먼저 입력 shape와 기대 출력 shape를 종이에 적은 뒤 아래 TODO를 구현한다.

In [ ]:
@dataclass
class DETRConfig:
    hidden_dim: int = 24
    heads: int = 4
    queries: int = 3
    num_classes: int = 4
    no_object_weight: float = 0.2
    box_weight: float = 5.0
    learning_rate: float = 0.015
    steps: int = 18


def box_l1_cost(predicted_boxes, target_boxes):
    return (predicted_boxes - target_boxes[:, None]).abs().sum(dim=-1)


def match_single_target(class_logits, predicted_boxes, labels, target_boxes):
    probabilities = class_logits.softmax(dim=-1)
    query_count = class_logits.shape[1]
    class_indices = labels[:, None, None].expand(-1, query_count, 1)
    class_cost = -probabilities.gather(2, class_indices).squeeze(-1)
    total_cost = class_cost + box_l1_cost(predicted_boxes, target_boxes)
    return total_cost.argmin(dim=1)


config = DETRConfig()
probe_logits = torch.zeros(2, 3, 5, device=DEVICE)
probe_boxes = torch.rand(2, 3, 4, device=DEVICE)
matched = match_single_target(probe_logits, probe_boxes, labels[:2], boxes[:2])
assert matched.shape == (2,)

### 구현 단계 2 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §3.2 Fig. 2: decoder object queries and image cross-attention
- **노트북에서 구현할 부분:** Task 2 explicit query self/cross-attention
- **구현 이유:** 논문의 계산 경로를 작은 tensor로 보존하면서 각 중간값을
  assertion으로 관찰하기 위해서다.
- **완료 증거:** [B,H,3,16] cross-attention weights

먼저 입력 shape와 기대 출력 shape를 종이에 적은 뒤 아래 TODO를 구현한다.

In [ ]:
class ExplicitMultiHeadAttention(nn.Module):
    """Q/K/V projection과 head 분할이 드러나는 attention이다."""

    def __init__(self, model_dim, heads):
        super().__init__()
        if model_dim % heads != 0:
            raise ValueError("model_dim must be divisible by heads")
        self.heads = heads
        self.head_dim = model_dim // heads
        self.query_projection = nn.Linear(model_dim, model_dim)
        self.key_projection = nn.Linear(model_dim, model_dim)
        self.value_projection = nn.Linear(model_dim, model_dim)
        self.output_projection = nn.Linear(model_dim, model_dim)

    def split_heads(self, tensor):
        batch, length, _ = tensor.shape
        tensor = tensor.reshape(
            batch,
            length,
            self.heads,
            self.head_dim,
        )
        return tensor.transpose(1, 2)

    def merge_heads(self, tensor):
        batch, _, length, _ = tensor.shape
        tensor = tensor.transpose(1, 2).contiguous()
        return tensor.reshape(
            batch,
            length,
            self.heads * self.head_dim,
        )

    def forward(self, query, key, value, attention_mask=None):
        queries = self.split_heads(self.query_projection(query))
        keys = self.split_heads(self.key_projection(key))
        values = self.split_heads(self.value_projection(value))
        scale = self.head_dim**-0.5
        scores = torch.matmul(queries, keys.transpose(-2, -1)) * scale
        if attention_mask is not None:
            scores = scores.masked_fill(
                attention_mask,
                float("-inf"),
            )
        weights = scores.softmax(dim=-1)
        context = torch.matmul(weights, values)
        output = self.output_projection(self.merge_heads(context))
        return output, weights


class DETRDecoderBlock(nn.Module):
    """Learned query를 self-attend한 뒤 image memory를 조회한다."""

    def __init__(self, config):
        super().__init__()
        hidden = config.hidden_dim
        self.self_attention = ExplicitMultiHeadAttention(
            hidden,
            config.heads,
        )
        self.cross_attention = ExplicitMultiHeadAttention(
            hidden,
            config.heads,
        )
        self.norm1 = nn.LayerNorm(hidden)
        self.norm2 = nn.LayerNorm(hidden)
        self.norm3 = nn.LayerNorm(hidden)
        self.feedforward = nn.Sequential(
            nn.Linear(hidden, hidden * 2),
            nn.ReLU(),
            nn.Linear(hidden * 2, hidden),
        )

    def forward(self, queries, memory):
        self_attended, _ = self.self_attention(
            queries,
            queries,
            queries,
        )
        queries = self.norm1(queries + self_attended)
        cross_attended, cross_weights = self.cross_attention(
            queries,
            memory,
            memory,
        )
        queries = self.norm2(queries + cross_attended)
        output = self.norm3(queries + self.feedforward(queries))
        return output, cross_weights


class DETRMini(nn.Module):
    def __init__(self, config):
        super().__init__()
        hidden = config.hidden_dim
        self.backbone = nn.Sequential(
            nn.Conv2d(1, hidden, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(hidden, hidden, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.memory_position = nn.Parameter(torch.randn(1, 16, hidden) * 0.02)
        self.decoder = DETRDecoderBlock(config)
        self.object_queries = nn.Parameter(torch.randn(config.queries, hidden) * 0.02)
        self.class_head = nn.Linear(hidden, config.num_classes + 1)
        self.box_head = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 4),
        )

    def encode(self, inputs):
        features = self.backbone(inputs)
        memory = features.flatten(2).transpose(1, 2)
        return memory + self.memory_position[:, : memory.shape[1]]

    def forward(self, inputs):
        memory = self.encode(inputs)
        queries = self.object_queries[None].expand(len(inputs), -1, -1)
        decoded, _ = self.decoder(queries, memory)
        return self.class_head(decoded), self.box_head(decoded).sigmoid()


model = DETRMini(config).to(DEVICE)
class_logits, predicted_boxes = model(images[:2])
memory = model.encode(images[:2])
queries = model.object_queries[None].expand(2, -1, -1)
_, cross_weights = model.decoder(queries, memory)
assert class_logits.shape == (2, 3, 5)
assert predicted_boxes.shape == (2, 3, 4)
assert cross_weights.shape == (2, 4, 3, 16)

### 구현 단계 3 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §3.1 Eq. (2) and §3.2: Hungarian set loss/no-object class
- **노트북에서 구현할 부분:** Task 3
- **구현 이유:** 논문의 계산 경로를 작은 tensor로 보존하면서 각 중간값을
  assertion으로 관찰하기 위해서다.
- **완료 증거:** set loss; GT-independent objectness query로 held-out class/box 평가

먼저 입력 shape와 기대 출력 shape를 종이에 적은 뒤 아래 TODO를 구현한다.

In [ ]:
def detr_set_loss(prediction, labels, target_boxes, config):
    class_logits, predicted_boxes = prediction
    matched = match_single_target(
        class_logits,
        predicted_boxes,
        labels,
        target_boxes,
    )
    no_object = config.num_classes
    class_targets = torch.full(
        class_logits.shape[:2],
        no_object,
        dtype=torch.long,
        device=labels.device,
    )
    batch_indices = torch.arange(len(labels), device=labels.device)
    class_targets[batch_indices, matched] = labels
    weights = class_logits.new_ones(config.num_classes + 1)
    weights[no_object] = config.no_object_weight
    class_loss = F.cross_entropy(
        class_logits.flatten(0, 1),
        class_targets.flatten(),
        weight=weights,
    )
    selected_boxes = predicted_boxes[batch_indices, matched]
    box_loss = F.l1_loss(selected_boxes, target_boxes)
    return class_loss + config.box_weight * box_loss


def train_detr_step(model, optimizer, inputs, labels, boxes, config):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    loss = detr_set_loss(model(inputs), labels, boxes, config)
    loss.backward()
    optimizer.step()
    return float(loss.detach())


def evaluate_detr(model, inputs, labels, boxes):
    model.eval()
    with torch.no_grad():
        class_logits, predicted_boxes = model(inputs)
        probabilities = class_logits.softmax(dim=-1)
        no_object_probability = probabilities[..., -1]
        selected = (1.0 - no_object_probability).argmax(dim=1)
        batch = torch.arange(len(labels), device=labels.device)
        selected_logits = class_logits[batch, selected, :-1]
        selected_boxes = predicted_boxes[batch, selected]
        predicted_labels = selected_logits.argmax(dim=-1)
        class_accuracy = (predicted_labels == labels).float().mean()
        box_l1 = F.l1_loss(selected_boxes, boxes)
        mean_objectness = (1.0 - no_object_probability[batch, selected]).mean()
    return {
        "objectness_selected_class_accuracy": float(class_accuracy),
        "objectness_selected_box_l1": float(box_l1),
        "selected_query_mean_objectness": float(mean_objectness),
    }


optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
losses = []
for step in range(config.steps):
    loss = train_detr_step(
        model,
        optimizer,
        x_train,
        y_train,
        boxes[train_idx],
        config,
    )
    losses.append(loss)
metrics = evaluate_detr(
    model,
    images[test_idx],
    labels[test_idx],
    boxes[test_idx],
)
assert min(losses[3:]) < losses[0]
accuracy = metrics["objectness_selected_class_accuracy"]
assert 0.0 <= accuracy <= 1.0
assert metrics["objectness_selected_box_l1"] >= 0.0
plt.figure(figsize=(5, 3))
plt.plot(losses)
plt.title(f"DETR objectness-selected class accuracy={accuracy:.1%}")
plt.tight_layout()
plt.show()
print(metrics)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> ResNet-50, 100 queries, COCO 대신 작은 CNN, 3 queries, 단일 물체를 사용한다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.